# XGBoost Classification
Student Success and Dropout Risk Prediction

Baseline multiclass classification using the existing stratified training and testing datasets.

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [ ]:
with open("../results/processed_data.pkl", "rb") as file:
    data = pickle.load(file)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

In [ ]:
encoder = LabelEncoder()

y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

print(dict(enumerate(encoder.classes_)))

In [ ]:
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(encoder.classes_),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train_encoded)

In [ ]:
xgb_pred_encoded = xgb.predict(X_test)
xgb_pred = encoder.inverse_transform(xgb_pred_encoded)
xgb_prob = xgb.predict_proba(X_test)

print(classification_report(y_test, xgb_pred))

In [ ]:
xgb_accuracy = accuracy_score(y_test, xgb_pred)

metrics = {
    "Model": "XGBoost",
    "Accuracy": xgb_accuracy,
    "Macro Precision": precision_score(
        y_test, xgb_pred, average="macro"
    ),
    "Macro Recall": recall_score(
        y_test, xgb_pred, average="macro"
    ),
    "Macro F1": f1_score(
        y_test, xgb_pred, average="macro"
    ),
    "Dropout Recall": recall_score(
        y_test == "Dropout",
        xgb_pred == "Dropout"
    ),
    "ROC-AUC": roc_auc_score(
        y_test_encoded,
        xgb_prob,
        multi_class="ovr",
        average="macro"
    )
}

pd.DataFrame([metrics])

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    xgb_pred,
    cmap="Blues"
)

plt.title("XGBoost Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance.head(15)

In [ ]:
importance.head(15).sort_values(
    "Importance"
).plot(
    kind="barh",
    x="Feature",
    y="Importance",
    figsize=(10, 7),
    legend=False
)

plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
pd.DataFrame([metrics]).to_csv(
    "../results/xgboost_metrics.csv",
    index=False
)

predictions = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": xgb_pred
})

predictions.to_csv(
    "../results/xgboost_predictions.csv",
    index=False
)

print("PASS: XGBoost results saved.")